# Layerize — flat image → ready-to-use layers (Colab GPU)
Run the cells top to bottom (**Runtime → Change runtime type → GPU**).  
The last cell prints a **URL + token**. Give those to the local MCP (`layerize_connect`) and your agent
(Claude Code / Codex) can send any image here and pull back transparent layers or a layered PSD for After Effects / Photoshop / Illustrator.  
Nothing big is ever downloaded to your PC — models live on this Colab VM.

In [ ]:
!nvidia-smi -L

In [ ]:
REPO = "https://github.com/danniedaniels209-bot/Nvidia-claude.git"   # private repo? use https://<TOKEN>@github.com/...
BRANCH = "claude/blissful-thompson-54b73y"
!rm -rf /content/layerize_src && git clone -q -b $BRANCH $REPO /content/layerize_src
!pip install -q git+https://github.com/facebookresearch/sam2.git "transformers>=4.45" opencv-python-headless pillow
print("installed")

In [ ]:
import sys, threading, time
sys.path.insert(0, "/content/layerize_src")
from layerize.engines import Engines
from layerize import server, tunnel

engines = Engines()
engines.warm()            # downloads SAM 2.1, ViTMatte, CLIP, LaMa onto the Colab VM (a few GB, one time)
httpd, token = server.serve(engines, host="127.0.0.1", port=8000, block=False)
url, _proc = tunnel.start(8000)

print("\n=== READY ===")
print("LAYERIZE_URL  =", url)
print("LAYERIZE_TOKEN=", token)
print("\nClaude Code:  claude mcp add layerize -e LAYERIZE_URL=%s -e LAYERIZE_TOKEN=%s -- python -m layerize.mcp_server" % (url, token))
print("(or just tell the agent to call layerize_connect with the URL and token)")
print("Keep this cell running; stopping it shuts the server down.")
while True:
    time.sleep(60)